# Challenge 3: Variational Quantum Eigensolver for LiH under a Resource Budget

Author: Abdullah Al Omar Galib (Ahkatlio)

| Item | Details |
|:--|:--|
| Difficulty | Hard |
| Required knowledge | Second quantization, Jordan-Wigner mapping, Pauli algebra, variational principle, stochastic optimization, noise channels |
| Required packages | `qiskit` 2.6, `qiskit-aer` 0.18, `numpy`, `scipy`, `matplotlib` (see `requirements.txt`). PySCF is optional. |
| Total available points | 100 |
| Objective | Estimate the ground-state energy of LiH with VQE on a noisy simulator within a fixed budget of qubits, two-qubit gates, energy evaluations and shots |
| Expected deliverables | Completed functions for Tasks VQE-T1 to VQE-T6, the experiments in Section 9, and written answers VQE-Q1 to VQE-Q4 |

## How to use this notebook

* Run the cells from top to bottom. The official grader re-executes the notebook in a clean, isolated environment with a fresh seed, and then calls your `solve_geometry` function for **undisclosed bond lengths**.
* Cells starting with `# Provided Code: Do Not Modify` are organizer-provided and must not be edited, deleted, duplicated or reordered.
* Cells starting with `# Participant Code: Task ...` are yours. Keep the function names and parameter lists exactly as given.
* Hints are collapsible and progressively more detailed. Using them does not affect your score.
* This challenge decides close rankings. Read the resource budget (Section 5.6) and the scoring formulas (Section 11) before you design your method.

## 1. Problem overview

The variational quantum eigensolver (VQE) is a hybrid algorithm: a quantum processor prepares a parameterized trial state and measures the energy, and a classical optimizer updates the parameters to minimize that energy. Because the energy of any trial state is an upper bound on the ground-state energy, VQE turns an eigenvalue problem into an optimization problem that only needs short circuits.

Here you will compute the ground-state energy of lithium hydride (LiH) along its dissociation curve. Hydrogen (H$_2$) is the usual first example, but a minimal-basis H$_2$ calculation needs only a single variational parameter, so it does not test much. LiH requires you to build and map a multi-orbital Hamiltonian, reduce it with symmetries, design an ansatz that is both expressive and shallow, and run a noisy, shot-limited optimization.

You work under a strict resource budget on a simulator with a fixed noise model. The challenge rewards careful choices: an ansatz with too few parameters cannot represent the ground state, while an ansatz with too many gates is dominated by noise and is hard to optimize with a limited number of shots.

## 2. Learning objectives

1. Build a second-quantized electronic Hamiltonian from molecular integrals and map it to qubits with the Jordan-Wigner transformation.
2. Use $\mathbb{Z}_2$ symmetries to remove qubits without changing the relevant part of the spectrum.
3. Estimate expectation values from measurement counts, with qubit-wise commuting measurement groups and correct standard errors.
4. Design an ansatz under a two-qubit gate budget and justify its expressivity.
5. Run and analyze a noisy, shot-limited optimization, and separate optimization error, shot noise and hardware noise in the final result.

## 3. Prerequisites

* Creation and annihilation operators, anticommutation relations and the occupation-number representation.
* Pauli strings and Qiskit's `SparsePauliOp` (`dot`, `simplify`, `to_matrix`, `group_commuting`).
* Parameterized circuits (`Parameter`, `ParameterVector`, `assign_parameters`).
* Basic numerical optimization and the effect of noisy function evaluations.

## 4. Scientific background

### 4.1 The electronic structure problem

Within the Born-Oppenheimer approximation, the electrons of a molecule with fixed nuclei are described by the electronic Hamiltonian. Expanding the electronic wavefunction in a finite set of orthonormal molecular orbitals gives the second-quantized Hamiltonian
$$
H = E_{\mathrm{core}} + \sum_{pq}\sum_{\sigma} h_{pq}\, a^{\dagger}_{p\sigma} a_{q\sigma}
+ \frac{1}{2}\sum_{pqrs}\sum_{\sigma\tau} (pq\vert rs)\, a^{\dagger}_{p\sigma} a^{\dagger}_{r\tau} a_{s\tau} a_{q\sigma},
$$
where $p, q, r, s$ label spatial orbitals, $\sigma, \tau \in \{\alpha, \beta\}$ label spins, $h_{pq}$ are one-electron integrals, $(pq\vert rs)$ are two-electron repulsion integrals in chemists' notation, and $E_{\mathrm{core}}$ collects the nuclear repulsion and the energy of frozen core electrons.

### 4.2 Active spaces

The cost of an exact (full configuration interaction) treatment grows exponentially with the number of orbitals. A standard approximation freezes chemically inert core orbitals and keeps only an *active space* of orbitals around the Fermi level, in which the electronic problem is solved exactly. Errors are then measured against the exact solution of the same active-space Hamiltonian.

### 4.3 The variational principle and VQE

For any normalized state $\lvert \psi \rangle$, $\langle \psi \rvert H \lvert \psi \rangle \ge E_0$, the ground-state energy. VQE minimizes $E(\boldsymbol{\theta}) = \langle \psi(\boldsymbol{\theta}) \rvert H \lvert \psi(\boldsymbol{\theta}) \rangle$ over the parameters of a circuit $U(\boldsymbol{\theta})$. On hardware, $E(\boldsymbol{\theta})$ is never known exactly: it is estimated from a finite number of shots, and gate noise prepares a mixed state $\rho(\boldsymbol{\theta})$ instead of $\lvert \psi(\boldsymbol{\theta}) \rangle$.

## 5. Mathematical formulation

### 5.1 Molecular system and approximations

| Quantity | Choice |
|:--|:--|
| Molecule | LiH, Li at the origin, H at $(0, 0, R)$ |
| Bond length $R$ | public values 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.6 and 3.0 angstrom; two undisclosed values in $[1.0, 3.0]$ are used for grading |
| Basis set | STO-3G |
| Charge and spin | neutral, singlet (multiplicity 1), restricted Hartree-Fock orbitals |
| Frozen core | the lowest $\sigma$ orbital (Li 1s), doubly occupied |
| Active space | the next three $\sigma$ (A$_1$ symmetry) orbitals with 2 electrons |
| Excluded orbitals | the two $\pi$ orbitals (E$_1$ symmetry), which do not mix with $\sigma$ orbitals |
| Qubit mapping | Jordan-Wigner, blocked spin ordering (below) |
| Reduction | tapering of the $\alpha$ and $\beta$ number parities (Task VQE-T2) |
| Reference energy $E_{\mathrm{ref}}$ | exact ground state of the active-space Hamiltonian with one $\alpha$ and one $\beta$ electron (equal to the CASCI(2,3) energy) |

The integrals were **precomputed** with PySCF 2.14.0 by the organizers and are distributed in `data/lih_sto3g_active_space_public.json`; you do not need PySCF. Each reference energy was computed in three independent ways (PySCF CASCI, PySCF FCI on the exported integrals, and exact diagonalization of the qubit Hamiltonian), which agree to better than $10^{-11}$ hartree. The active-space approximation itself differs from the full STO-3G FCI energy by 0.5 to 2.2 millihartree over the published geometries; this difference is not part of the challenge.

### 5.2 Jordan-Wigner mapping

With $n = 3$ active spatial orbitals there are $2n = 6$ spin orbitals. Use the **blocked ordering**: qubit $p$ represents orbital $p$ with spin $\alpha$ and qubit $p + n$ represents orbital $p$ with spin $\beta$ ($p = 0, 1, 2$). The Jordan-Wigner transformation maps
$$
a^{\dagger}_{j} = \Big(\prod_{k<j} Z_k\Big)\, \frac{X_j - iY_j}{2}, \qquad
a_{j} = \Big(\prod_{k<j} Z_k\Big)\, \frac{X_j + iY_j}{2}.
$$
In this ordering the Hartree-Fock state, with orbital 0 occupied for both spins, is the computational basis state with qubits 0 and 3 set to $\lvert 1 \rangle$.

### 5.3 Symmetries and tapering

$H$ conserves the numbers of $\alpha$ and $\beta$ electrons, so the parity operators $P_\alpha = Z_0 Z_1 Z_2$ and $P_\beta = Z_3 Z_4 Z_5$ commute with $H$. Following Bravyi et al. (2017), each such $\mathbb{Z}_2$ symmetry can be rotated by a Clifford transformation onto a single qubit, which is then fixed to the eigenvalue of the desired sector and removed. The reduced Hamiltonian acts on 4 qubits and its spectrum is the part of the spectrum of $H$ in the chosen sector. For all published and evaluation geometries the organizers verified numerically that the lowest eigenvalue of the full 6-qubit Hamiltonian, over all particle numbers, is $E_{\mathrm{ref}}$, and that the same holds for the tapered Hamiltonian in the Hartree-Fock sector. Consequently no trial state can reach an energy below $E_{\mathrm{ref}}$.

### 5.4 VQE objective and error measures

$$
E(\boldsymbol{\theta}) = \langle \psi(\boldsymbol{\theta}) \rvert H \lvert \psi(\boldsymbol{\theta}) \rangle \ge E_{\mathrm{ref}},
\qquad
\Delta E = \left| E_{\mathrm{VQE}} - E_{\mathrm{ref}} \right| .
$$
Energies are reported in hartree ($E_{\mathrm{h}}$) and errors in millihartree ($\mathrm{m}E_{\mathrm{h}}$). Chemical accuracy is conventionally taken as $1\ \mathrm{kcal/mol} \approx 1.6 \times 10^{-3}\,E_{\mathrm{h}}$. It is a benchmark target, not a guaranteed outcome under the budget below. Three different energies of the final state must not be confused:

* the **ideal energy** $E_{\mathrm{ideal}} = \langle \psi(\boldsymbol{\theta}^{*}) \rvert H \lvert \psi(\boldsymbol{\theta}^{*}) \rangle$ of the noiseless state, which measures the quality of the parameters you found;
* the **noisy energy** $E_{\mathrm{noisy}} = \operatorname{Tr}\big[\rho(\boldsymbol{\theta}^{*}) H\big]$ of the state actually prepared under the noise model, which measures how well the circuit tolerates noise;
* the **finite-shot estimate** $\hat{E}$ with standard error $\sigma_{\hat{E}}$, which is a random variable with mean $E_{\mathrm{noisy}}$.

### 5.5 Measurement

The sampler measures every qubit in a Pauli basis. Terms that are qubit-wise commuting (on each qubit the two Pauli letters are equal or one of them is $I$) can be measured with the same basis. For a group $G = \sum_k c_k P_k$ measured with $m$ shots, every shot with bit string $b$ gives the value
$$
v(b) = \sum_k c_k \,(-1)^{\sum_{q \in \operatorname{supp}(P_k)} b_q},
$$
and $\langle G \rangle$ is estimated by the sample mean $\bar{v}$ with standard error $\sqrt{\widehat{\operatorname{Var}}(v)/m}$. Different groups are measured on independent shots, so their variances add.

### 5.6 Noise model and resource budget

**Noise model** (`ph.vqe_noise_model()`): two-qubit depolarizing error with parameter $0.002$ after every `cx`, single-qubit depolarizing error with parameter $0.0002$ after every `sx` and `x`, ideal `rz`, ideal measurement basis changes and ideal readout. Circuits are compiled with `ph.official_transpile` (basis `rz`, `sx`, `x`, `cx`; all-to-all connectivity; optimization level 1).

**Budget for one call of `solve_geometry`** (`ph.VQE_BUDGET`), enforced by `ph.BudgetedSampler` and verified again by the grader:

| Resource | Limit | What counts |
|:--|--:|:--|
| Qubits per circuit | 6 | `circuit.num_qubits` |
| Two-qubit gates per circuit | 40 | gates on two or more qubits after `ph.official_transpile` |
| Energy evaluations (sampler jobs) | 600 | calls of `sampler.run`; this is the limit on optimizer iterations (an optimizer with two evaluations per iteration, such as SPSA, can run at most 300 iterations) |
| Circuit executions | 20 000 | one per measurement basis per job |
| Total shots | 20 000 000 | shots per basis times number of bases, summed over jobs |
| Wall-clock time | 300 s | measured from the creation of the sampler |

**What is not allowed inside `solve_geometry`:** computing energies, gradients or states by any route other than `sampler.run` (for example `Statevector`, `DensityMatrix`, `ph.noisy_density_matrix`, exact diagonalization, or a separate `AerSimulator`), or using precomputed parameters for specific bond lengths. Exact methods are allowed elsewhere in the notebook for validation and analysis, as in the public checks below.

## 6. Environment checks

In [ ]:
# Provided Code: Do Not Modify
import sys
from pathlib import Path

_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "public_helpers.py").exists()), None)
if _ROOT is None:
    raise FileNotFoundError("Could not find src/public_helpers.py. Start Jupyter from inside the hackathon folder.")
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))

import itertools
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import scipy
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter, ParameterVector
from qiskit.circuit.library import XXPlusYYGate
from qiskit.quantum_info import SparsePauliOp, Statevector, Z2Symmetries

import public_helpers as ph
from public_helpers import BudgetedSampler, BudgetExceededError, VQE_BUDGET, count_two_qubit_gates

ph.check_environment()

## 7. Provided starter code

* `ph.load_public_integrals(bond_length)` returns a dictionary with `h1` (shape $(3,3)$), `eri` (shape $(3,3,3,3)$, chemists' notation), `e_core`, `n_orbitals`, `n_electrons`, and the public reference values `e_hf` and `e_ref`. Hidden evaluation geometries are passed to your code in the same format **without** `e_hf` and `e_ref`.
* `BudgetedSampler(seed=...)` is the only permitted way to obtain measurement data inside `solve_geometry`. Its method `run(circuit, parameter_values, bases, shots)` prepares the state of a measurement-free, possibly parameterized circuit under the official noise model, measures it in every basis of `bases` (strings over `XYZI` in Qiskit order) and returns one counts dictionary per basis. Parameter values follow the order of `circuit.parameters`. The attributes `usage`, `remaining` and `ledger` report the resources used.
* `ph.noisy_density_matrix(circuit, parameter_values)` returns the exact noisy state, for analysis outside `solve_geometry` only.
* `count_two_qubit_gates(circuit)` returns the official two-qubit gate count.

In [ ]:
# Provided Code: Do Not Modify
SEED = ph.get_seed()
PUBLIC_DATA = ph.load_public_integrals()
BOND_LENGTHS = [entry["bond_length_angstrom"] for entry in PUBLIC_DATA]
EXAMPLE = ph.load_public_integrals(1.6)

print(f"Session seed: {SEED}")
print(f"Public bond lengths (angstrom): {BOND_LENGTHS}")
print(f"Budget: {VQE_BUDGET}")
print(f"R = 1.6 angstrom: E_HF = {EXAMPLE['e_hf']:.6f} Ha, E_ref = {EXAMPLE['e_ref']:.6f} Ha, "
      f"correlation energy = {1000 * (EXAMPLE['e_ref'] - EXAMPLE['e_hf']):.2f} mHa")

## 8. Participant implementation tasks

Tasks VQE-T1 to VQE-T4 are building blocks with exact answers. Tasks VQE-T5 and VQE-T6 are open-ended: you choose the ansatz and the optimizer, and your result is scored by its accuracy at the undisclosed geometries (Section 11).

### Task VQE-T1: Qubit Hamiltonian (15 points)

Implement `build_qubit_hamiltonian(integrals)`.

* **Input:** a dictionary with the keys `h1`, `eri` and `e_core` (and possibly others) for $n$ active orbitals; arrays may be NumPy arrays or nested lists.
* **Returns:** a `SparsePauliOp` on $2n$ qubits equal to the Jordan-Wigner image of the Hamiltonian of Section 4.1, with the blocked spin ordering of Section 5.2. The identity term must include $E_{\mathrm{core}}$. Combine duplicate Pauli strings and drop numerically zero terms (`simplify`). The coefficients of a Hermitian operator are real; return them with zero imaginary part.
* **Edge cases:** raise `ValueError` if the integral shapes are inconsistent.
* **Evaluation:** the grader compares your operator with the reference operator for a public geometry and two undisclosed geometries (tolerance $10^{-8}$). A different but valid spin-orbital ordering, detected through an identical spectrum, earns 60 percent of the points.

In [ ]:
# Participant Code: Task VQE-T1
def build_qubit_hamiltonian(integrals: dict) -> SparsePauliOp:
    """Jordan-Wigner qubit Hamiltonian of the active space (blocked spin ordering)."""
    h1 = np.asarray(integrals["h1"], dtype=float)
    eri = np.asarray(integrals["eri"], dtype=float)
    if h1.ndim != 2 or h1.shape[0] != h1.shape[1]:
        raise ValueError("h1 must be a square matrix")
    n = h1.shape[0]
    if eri.shape != (n, n, n, n):
        raise ValueError("eri must have shape (n, n, n, n)")
    n_qubits = 2 * n                                   # qubit p: orbital p, spin alpha; qubit p + n: spin beta

    def annihilation(j):
        # a_j = Z_0 ... Z_{j-1} (X_j + i Y_j) / 2
        string, qubits = "Z" * j, list(range(j))
        return (SparsePauliOp.from_sparse_list([(string + "X", qubits + [j], 0.5)], num_qubits=n_qubits)
                + SparsePauliOp.from_sparse_list([(string + "Y", qubits + [j], 0.5j)], num_qubits=n_qubits))

    a = [annihilation(j) for j in range(n_qubits)]
    ad = [op.adjoint() for op in a]
    orbital = lambda p, spin: p + spin * n

    hamiltonian = float(integrals["e_core"]) * SparsePauliOp.from_sparse_list([("", [], 1.0)], num_qubits=n_qubits)
    for p, q in itertools.product(range(n), repeat=2):
        if h1[p, q] != 0.0:
            for spin in (0, 1):
                hamiltonian = hamiltonian + h1[p, q] * (ad[orbital(p, spin)] @ a[orbital(q, spin)])
    for p, q, r, s in itertools.product(range(n), repeat=4):
        value = eri[p, q, r, s]
        if abs(value) < 1e-14:
            continue
        for sigma, tau in itertools.product((0, 1), repeat=2):   # (pq|rs) a+_{p sigma} a+_{r tau} a_{s tau} a_{q sigma}
            hamiltonian = hamiltonian + 0.5 * value * (ad[orbital(p, sigma)] @ ad[orbital(r, tau)]
                                                       @ a[orbital(s, tau)] @ a[orbital(q, sigma)])
    hamiltonian = hamiltonian.simplify(atol=1e-12)
    return SparsePauliOp(hamiltonian.paulis, np.real(hamiltonian.coeffs))

#### Hints for Task VQE-T1

Open the hints one at a time. Each hint reveals more than the previous one.

<details>
<summary>Reveal Hint 1 (Conceptual direction)</summary>

Build the creation and annihilation operators of all $2n$ spin orbitals as qubit operators first; the Hamiltonian is then a sum of their products weighted by the integrals.

</details>

<details>
<summary>Reveal Hint 2 (Mathematical guidance)</summary>

The two-electron term with chemists' integrals is $\frac{1}{2}\sum (pq\vert rs)\, a^{\dagger}_{p\sigma} a^{\dagger}_{r\tau} a_{s\tau} a_{q\sigma}$ with the operator order exactly as written. Terms with $p\sigma = r\tau$ or $q\sigma = s\tau$ vanish because $(a^{\dagger}_{j})^2 = a_j^2 = 0$. In Qiskit labels qubit $k$ is the character at position $2n - 1 - k$.

</details>

<details>
<summary>Reveal Hint 3 (Implementation guidance)</summary>

Represent $a^{\dagger}_j$ as `SparsePauliOp([x_label, y_label], [0.5, -0.5j])` with $Z$ on all qubits below $j$. Use `A.dot(B)` for the matrix product $AB$, collect the terms in a list, and finish with `SparsePauliOp.sum(terms).simplify(atol=1e-12)`. Check your result: the lowest eigenvalue in the sector with one electron of each spin must equal `e_ref`.

</details>

In [ ]:
# Provided Code: Do Not Modify
def _sector_ground_energy(hamiltonian, n_alpha, n_beta, n_orbitals):
    matrix = hamiltonian.to_matrix()
    mask = (1 << n_orbitals) - 1
    indices = [i for i in range(2 ** (2 * n_orbitals))
               if bin(i & mask).count("1") == n_alpha and bin(i >> n_orbitals).count("1") == n_beta]
    return float(np.linalg.eigvalsh(matrix[np.ix_(indices, indices)])[0])

def _public_check_t1():
    hamiltonian = build_qubit_hamiltonian(EXAMPLE)
    assert isinstance(hamiltonian, SparsePauliOp) and hamiltonian.num_qubits == 6, "expected 6 qubits"
    matrix = hamiltonian.to_matrix()
    assert np.allclose(matrix, matrix.conj().T), "the Hamiltonian must be Hermitian"
    energy = _sector_ground_energy(hamiltonian, 1, 1, 3)
    assert abs(energy - EXAMPLE["e_ref"]) < 1e-8, f"sector ground energy {energy:.8f} != {EXAMPLE['e_ref']:.8f}"
    print(f"  {len(hamiltonian)} Pauli terms; ground energy {energy:.8f} Ha")

ph.run_public_check("VQE-T1", _public_check_t1)

### Task VQE-T2: Symmetry reduction (10 points)

Implement `taper_hamiltonian(hamiltonian, n_alpha=1, n_beta=1)`.

* **Input:** the 6-qubit Hamiltonian of Task VQE-T1 and the numbers of $\alpha$ and $\beta$ electrons of the Hartree-Fock state.
* **Returns:** a `SparsePauliOp` on fewer qubits (4 for this system) obtained by tapering the $\alpha$- and $\beta$-parity symmetries, in the symmetry sector that contains the Hartree-Fock state.
* **Evaluation:** for undisclosed geometries the grader checks that your operator has fewer than 6 qubits, that every eigenvalue is an eigenvalue of the full Hamiltonian, and that its lowest eigenvalue equals $E_{\mathrm{ref}}$ (tolerance $10^{-6}$ hartree).

In [ ]:
# Participant Code: Task VQE-T2
def taper_hamiltonian(hamiltonian: SparsePauliOp, n_alpha: int = 1, n_beta: int = 1) -> SparsePauliOp:
    """Remove the qubits fixed by the alpha and beta parity symmetries (Hartree-Fock sector)."""
    n_orbitals = hamiltonian.num_qubits // 2
    symmetries = Z2Symmetries.find_z2_symmetries(hamiltonian)
    occupied = set(range(n_alpha)) | set(range(n_orbitals, n_orbitals + n_beta))   # Hartree-Fock qubits
    sector = []
    for generator in symmetries.symmetries:
        label = generator.to_label()                   # last character = qubit 0
        z_on_occupied = sum(1 for q in occupied if label[-1 - q] == "Z")
        sector.append(-1 if z_on_occupied % 2 else 1)  # eigenvalue of the generator on the HF state
    tapering = Z2Symmetries(symmetries.symmetries, symmetries.sq_paulis, symmetries.sq_list,
                            tapering_values=sector)
    tapered = tapering.taper(hamiltonian)
    return tapered

#### Hints for Task VQE-T2

Open the hints one at a time. Each hint reveals more than the previous one.

<details>
<summary>Reveal Hint 1 (Conceptual direction)</summary>

Each symmetry generator is a Pauli string that commutes with every term of $H$. Fixing its eigenvalue removes one qubit. The correct eigenvalue is the one taken by the Hartree-Fock state.

</details>

<details>
<summary>Reveal Hint 2 (Mathematical guidance)</summary>

For a $Z$-type generator $\tau = \prod_{q \in S} Z_q$ and a computational basis state, the eigenvalue is $(-1)^{\text{number of occupied qubits in } S}$. The Hartree-Fock state has qubits $0, \ldots, n_\alpha - 1$ and $n, \ldots, n + n_\beta - 1$ occupied.

</details>

<details>
<summary>Reveal Hint 3 (Implementation guidance)</summary>

`Z2Symmetries.find_z2_symmetries(hamiltonian)` finds the generators (attribute `symmetries`, each a `Pauli` with boolean arrays `z` and `x`). Create `Z2Symmetries(z2.symmetries, z2.sq_paulis, z2.sq_list, tapering_values=[...])` with your eigenvalues and call its `taper` method.

</details>

In [ ]:
# Provided Code: Do Not Modify
def _public_check_t2():
    tapered = taper_hamiltonian(build_qubit_hamiltonian(EXAMPLE))
    assert isinstance(tapered, SparsePauliOp) and tapered.num_qubits < 6, "the tapered operator must have fewer qubits"
    energy = float(np.linalg.eigvalsh(tapered.to_matrix())[0])
    assert abs(energy - EXAMPLE["e_ref"]) < 1e-6, f"lowest eigenvalue {energy:.8f} != {EXAMPLE['e_ref']:.8f}"
    print(f"  {tapered.num_qubits} qubits, {len(tapered)} Pauli terms")

ph.run_public_check("VQE-T2", _public_check_t2)

### Task VQE-T3: Measurement groups (8 points)

Implement `group_qubitwise_commuting(hamiltonian)`.

* **Returns:** a list of pairs `(basis, group)`. `group` is a `SparsePauliOp` containing some of the terms of `hamiltonian`; `basis` is a string of length `num_qubits` over `XYZ` in Qiskit order such that every non-identity Pauli letter of every term of the group equals the basis letter on that qubit. Use `"Z"` on qubits where all terms of the group act trivially.
* **Requirements:** every term of `hamiltonian` (including the identity) appears in exactly one group, so that the groups sum to `hamiltonian`.
* **Evaluation:** validity and completeness on undisclosed Hamiltonians. The number of groups is not scored directly, but fewer groups mean fewer shots per energy evaluation.

In [ ]:
# Participant Code: Task VQE-T3
def group_qubitwise_commuting(hamiltonian: SparsePauliOp) -> list:
    """Return a list of (basis, group) pairs that partition the Hamiltonian."""
    labels = hamiltonian.paulis.to_labels()
    coeffs = hamiltonian.coeffs

    def greedy(order):
        groups = []                                    # each group: per-qubit letter (None = free) and term indices
        for i in order:
            label = labels[i]
            for group in groups:
                if all(c == "I" or letter in (None, c) for c, letter in zip(label, group["letters"])):
                    group["letters"] = [letter if c == "I" else c for c, letter in zip(label, group["letters"])]
                    group["terms"].append(i)
                    break
            else:
                groups.append({"letters": [None if c == "I" else c for c in label], "terms": [i]})
        return groups

    weight = lambda i: sum(c != "I" for c in labels[i])
    indices = range(len(labels))
    orders = [sorted(indices, key=lambda i: (-weight(i), -abs(coeffs[i]))),   # heaviest terms first
              sorted(indices, key=lambda i: -abs(coeffs[i])),
              sorted(indices, key=lambda i: (-weight(i), i))]
    best = min((greedy(order) for order in orders), key=len)                 # keep the smallest partition
    return [("".join("Z" if letter is None else letter for letter in group["letters"]),
             SparsePauliOp([labels[i] for i in group["terms"]], [coeffs[i] for i in group["terms"]]))
            for group in best]

#### Hints for Task VQE-T3

Open the hints one at a time. Each hint reveals more than the previous one.

<details>
<summary>Reveal Hint 1 (Conceptual direction)</summary>

Two Pauli strings can share a measurement basis if, on every qubit, they carry the same letter or one of them carries $I$.

</details>

<details>
<summary>Reveal Hint 2 (Mathematical guidance)</summary>

Finding the minimum number of groups is a graph-colouring problem (Verteletskyi et al., 2020); a greedy or library heuristic is sufficient. The basis of a group follows from the non-identity letters of its members.

</details>

<details>
<summary>Reveal Hint 3 (Implementation guidance)</summary>

`hamiltonian.group_commuting(qubit_wise=True)` returns a list of `SparsePauliOp` groups. Build each basis string from `group.paulis.to_labels()`.

</details>

In [ ]:
# Provided Code: Do Not Modify
def _public_check_t3():
    hamiltonian = build_qubit_hamiltonian(EXAMPLE)
    groups = group_qubitwise_commuting(hamiltonian)
    total = SparsePauliOp.sum([group for _, group in groups])
    assert np.allclose((total - hamiltonian).simplify(atol=1e-12).coeffs, 0, atol=1e-10), "groups do not sum to H"
    for basis, group in groups:
        for label in group.paulis.to_labels():
            assert all(c == "I" or c == b for c, b in zip(label, basis)), f"{label} not measurable in {basis}"
    print(f"  {len(groups)} measurement groups for {len(hamiltonian)} terms")

ph.run_public_check("VQE-T3", _public_check_t3)

### Task VQE-T4: Expectation values from counts (7 points)

Implement two functions.

1. `expectation_from_counts(group, counts) -> tuple` returns `(mean, standard_error)` of $\langle G \rangle$ from a counts dictionary measured in the basis of the group, using the per-shot estimator of Section 5.5. Raise `ValueError` for empty counts or bit strings of the wrong length.
2. `estimate_energy(hamiltonian, ansatz, parameters, sampler, shots) -> tuple` measures all groups of `hamiltonian` in **one** call of `sampler.run` and returns `(energy, standard_error)`.

**Evaluation:** `expectation_from_counts` is compared with exact values on hidden synthetic counts (tolerance $10^{-9}$). `estimate_energy` is used by your own `solve_geometry`.

In [ ]:
# Participant Code: Task VQE-T4
def expectation_from_counts(group: SparsePauliOp, counts: dict) -> tuple:
    """Return (mean, standard_error) of <group> from counts in the group's basis."""
    n_qubits = group.num_qubits
    total = sum(counts.values()) if counts else 0
    if total <= 0:
        raise ValueError("counts are empty")
    labels = group.paulis.to_labels()
    coeffs = np.real(group.coeffs)
    support = np.array([[c != "I" for c in label[::-1]] for label in labels], dtype=int)   # [term, qubit]
    strings = list(counts)
    if any(len(s) != n_qubits for s in strings):
        raise ValueError("bit strings must have one character per qubit")
    bits = np.array([[int(c) for c in s[::-1]] for s in strings], dtype=int)               # [outcome, qubit]
    values = (1 - 2 * ((bits @ support.T) % 2)) @ coeffs                                     # v(b) per outcome
    weights = np.array([counts[s] for s in strings], dtype=float)
    mean = float(weights @ values / total)
    variance = float(weights @ (values - mean) ** 2 / total)
    return mean, math.sqrt(variance / total)


_GROUP_CACHE = {}


def estimate_energy(hamiltonian: SparsePauliOp, ansatz: QuantumCircuit, parameters, sampler, shots: int) -> tuple:
    """Return (energy, standard_error) from a single sampler job."""
    key = (tuple(hamiltonian.paulis.to_labels()), tuple(np.real(hamiltonian.coeffs)))
    if key not in _GROUP_CACHE:
        _GROUP_CACHE[key] = group_qubitwise_commuting(hamiltonian)
    groups = _GROUP_CACHE[key]
    counts = sampler.run(ansatz, parameters, [basis for basis, _ in groups], shots)     # one job for all groups
    means, variances = 0.0, 0.0
    for (_, group), group_counts in zip(groups, counts):
        mean, error = expectation_from_counts(group, group_counts)
        means += mean
        variances += error ** 2
    return float(means), math.sqrt(variances)

#### Hints for Task VQE-T4

Open the hints one at a time. Each hint reveals more than the previous one.

<details>
<summary>Reveal Hint 1 (Conceptual direction)</summary>

After the basis change, every Pauli letter in a group becomes $Z$ on its qubit, and $Z$ has eigenvalue $(-1)^{b_q}$ for bit $b_q$. A product of $Z$ operators has the product of the eigenvalues.

</details>

<details>
<summary>Reveal Hint 2 (Mathematical guidance)</summary>

Treat each shot as one sample of $v(b)$. The terms of a group are measured on the same shots, so their estimates are correlated; computing the variance of $v$ directly accounts for this, whereas adding the variances of individual terms does not.

</details>

<details>
<summary>Reveal Hint 3 (Implementation guidance)</summary>

In a Qiskit bit string the character at position $i$ refers to qubit $\text{len} - 1 - i$, exactly as in Pauli labels, so you can compare `label[i]` and `bits[i]` directly. Use weighted means with the counts as weights.

</details>

In [ ]:
# Provided Code: Do Not Modify
def _public_check_t4():
    mean, error = expectation_from_counts(SparsePauliOp(["Z"], [1.0]), {"0": 75, "1": 25})
    assert abs(mean - 0.5) < 1e-12, f"mean {mean} != 0.5"
    assert abs(error - math.sqrt(0.75 / 100)) < 1e-12, f"standard error {error} is wrong"
    # "01": ZZ -> -1, IZ -> -1, value -1.5; "11": ZZ -> +1, IZ -> -1, value 0.5
    mean, _ = expectation_from_counts(SparsePauliOp(["ZZ", "IZ"], [1.0, 0.5]), {"01": 30, "11": 10})
    assert abs(mean - (-1.0)) < 1e-12, f"mean {mean} != -1.0 for a two-term group"

ph.run_public_check("VQE-T4", _public_check_t4)

### Task VQE-T5: Ansatz (10 points)

Implement `build_ansatz(num_qubits)`.

* **Input:** the number of qubits of the Hamiltonian you optimize (6 if you do not taper, 4 if you do).
* **Returns:** a parameterized `QuantumCircuit` on `num_qubits` qubits without measurements, resets or classical bits. It must prepare a reasonable initial state (for example Hartree-Fock) for some parameter values.
* **Budget:** at most 40 two-qubit gates after `ph.official_transpile`. Every additional two-qubit gate adds noise; Section 5.4 explains how this affects your score.
* **Evaluation:** the circuits returned by `solve_geometry` at the undisclosed geometries are checked for validity and for the gate budget.

Justify your choice in answer VQE-Q2. Physically motivated ansatze (particle- and spin-conserving, built from Givens rotations or excitation operators) and hardware-efficient ansatze are both acceptable.

In [ ]:
# Participant Code: Task VQE-T5
# Natural-orbital ansatz for 1 alpha + 1 beta electron in 3 orbitals (6 qubits, JW blocked ordering).
# The exact singlet is sum_ij C_ij |i_alpha j_beta> with C real symmetric = U diag(s) U^T. We prepare
# sum_k s_k |k_alpha k_beta> (2 Givens rotations + 3 CNOTs) and rotate both spin species by the same
# orbital rotation U (3 Givens rotations per spin): 5 parameters, enough for every real singlet.
# E(theta_k) is a trigonometric polynomial with frequencies f0, 2 f0, ..., K f0 (found by scanning each
# parameter with Statevector): (f0, K) per parameter. Half-angle terms (f0 = 1/2) come from the Givens gates.
_ANSATZ_FREQUENCIES = [(1.0, 1), (0.5, 2), (1.0, 2), (0.5, 4), (0.5, 4)]


def build_ansatz(num_qubits: int) -> QuantumCircuit:
    """Parameterized trial circuit for a Hamiltonian on num_qubits qubits."""
    if num_qubits != 6:
        raise ValueError("this ansatz is defined for the 6-qubit Hamiltonian")
    theta = ParameterVector("theta", 5)
    real_rotation = math.pi / 2                       # phase of the Givens gate that keeps amplitudes real
    circuit = QuantumCircuit(6)
    circuit.x(0)                                      # Hartree-Fock: orbital 0 doubly occupied after the copy below
    circuit.append(XXPlusYYGate(theta[0], real_rotation), [0, 1])     # spread the alpha electron over 3 orbitals
    circuit.append(XXPlusYYGate(theta[1], real_rotation), [1, 2])
    for k in range(3):
        circuit.cx(k, k + 3)                          # beta electron follows the alpha electron
    for angle, (i, j) in zip(theta[2:], [(0, 1), (1, 2), (0, 1)]):    # orbital rotation, same for both spins
        circuit.append(XXPlusYYGate(angle, real_rotation), [i, j])
        circuit.append(XXPlusYYGate(angle, real_rotation), [i + 3, j + 3])
    return circuit

#### Hints for Task VQE-T5

Open the hints one at a time. Each hint reveals more than the previous one.

<details>
<summary>Reveal Hint 1 (Conceptual direction)</summary>

Start from the Hartree-Fock state and add only the correlations that matter. With two electrons in a singlet, the exact ground state is $\sum_{ij} C_{ij} \lvert i_\alpha j_\beta \rangle$ with a symmetric $3 \times 3$ matrix $C$, so only a handful of real parameters are needed.

</details>

<details>
<summary>Reveal Hint 2 (Mathematical guidance)</summary>

A symmetric $C$ can be diagonalized, $C = U \operatorname{diag}(c) U^{T}$: the state is $\sum_k c_k \lvert k_\alpha k_\beta \rangle$ written in rotated orbitals. Pair amplitudes $c_k$ and orbital rotations $U$ can each be implemented with Givens rotations, which act as real rotations in the span of $\lvert 01 \rangle$ and $\lvert 10 \rangle$. After tapering, work out which 4-qubit basis states correspond to which determinants: the Hartree-Fock state becomes $\lvert 0000 \rangle$.

</details>

<details>
<summary>Reveal Hint 3 (Implementation guidance)</summary>

`XXPlusYYGate(2 * t, -math.pi / 2)` is a real Givens rotation by angle `t` on two qubits that costs two CNOTs after transpilation. `ry` and `cx` gates are cheap ways to create and copy amplitudes. Check expressivity first without noise: minimize the exact energy of your circuit with `scipy.optimize.minimize` and `Statevector` in an exploration cell, and compare with `e_ref`.

</details>

In [ ]:
# Provided Code: Do Not Modify
def _public_check_t5():
    for n_qubits in (4, 6):
        try:
            ansatz = build_ansatz(n_qubits)
        except ValueError:
            continue
        assert ansatz.num_clbits == 0, "the ansatz must not contain classical bits or measurements"
        assert ansatz.num_parameters > 0, "the ansatz must have parameters"
        gates = count_two_qubit_gates(ansatz)
        assert gates <= VQE_BUDGET.max_two_qubit_gates, f"{gates} two-qubit gates exceed the budget"
        print(f"  {n_qubits} qubits: {ansatz.num_parameters} parameters, {gates} two-qubit gates")
        return
    raise AssertionError("build_ansatz did not accept 4 or 6 qubits")

ph.run_public_check("VQE-T5", _public_check_t5)

### Task VQE-T6: Budgeted optimization and the complete workflow (40 points in total, see Section 11)

Implement `run_vqe(hamiltonian, sampler, seed)` and `solve_geometry(integrals, sampler, seed)`.

* `run_vqe` minimizes the energy of your ansatz for the given Hamiltonian, using only `sampler.run` (through `estimate_energy`) to obtain energies.
* `solve_geometry` runs the complete workflow for one geometry: Hamiltonian (VQE-T1), optional tapering (VQE-T2), and `run_vqe`. The grader calls it with a fresh sampler for each undisclosed geometry, and the integrals dictionary contains no reference energies.
* **Both functions return** a dictionary with the keys:

| Key | Type | Meaning |
|:--|:--|:--|
| `"ansatz"` | `QuantumCircuit` | parameterized circuit (no measurements) |
| `"parameters"` | 1D array of floats | final parameters, in the order of `ansatz.parameters` |
| `"energy"` | `float` | final finite-shot estimate $\hat{E}$ in hartree, obtained from the sampler at the final parameters without error mitigation |
| `"std_error"` | `float` | standard error of `"energy"` |
| `"hamiltonian"` | `SparsePauliOp` | the Hamiltonian that was optimized (6-qubit or tapered) |
| `"history"` | list of floats | energy estimates during the optimization |

You may add further keys, for example `"mitigated_energy"` if you implement error mitigation; it is discussed in the manual review but not scored automatically.

* **Requirements:** respect the budget of Section 5.6 (a `BudgetExceededError` ends the run without a result), evaluate the final parameters with the sampler at least once, and finish within the 300 s time limit. Use `seed` for every random choice.

In [ ]:
# Participant Code: Task VQE-T6
def _fit_minimum(offsets, values, base, n_harmonics):
    """Fit E(phi) = a0 + sum_h (a_h cos(h base phi) + b_h sin(h base phi)); return (argmin, min value)."""
    design = lambda phi: np.column_stack([np.ones_like(phi)] + [f(h * base * phi) for h in range(1, n_harmonics + 1)
                                                                  for f in (np.cos, np.sin)])
    coeffs = np.linalg.lstsq(design(np.asarray(offsets)), np.asarray(values), rcond=None)[0]
    half_period = math.pi / base
    grid = np.linspace(-half_period, half_period, 2001)
    curve = design(grid) @ coeffs
    k = int(np.argmin(curve))
    lo, hi = grid[max(k - 1, 0)], grid[min(k + 1, len(grid) - 1)]
    fine = np.linspace(lo, hi, 201)                                  # refine around the best grid point
    fine_curve = design(fine) @ coeffs
    j = int(np.argmin(fine_curve))
    return float(fine[j]), float(fine_curve[j])


N_SWEEPS, SHOT_GROWTH = 6, 1.6      # optimizer settings: number of sweeps and shot growth per sweep
FINAL_RESERVE = 0.10                # fraction of the shot budget kept for the final high-precision estimate


def run_vqe(hamiltonian: SparsePauliOp, sampler: BudgetedSampler, seed: int) -> dict:
    """Minimize the energy of the ansatz with the budgeted sampler."""
    rng = np.random.default_rng(seed)
    start = time.monotonic()
    ansatz = build_ansatz(hamiltonian.num_qubits)
    n_params = ansatz.num_parameters
    frequencies = _ANSATZ_FREQUENCIES
    n_groups = len(group_qubitwise_commuting(hamiltonian))
    total_shots = sampler.remaining["total_shots"]
    final_reserve = int(FINAL_RESERVE * total_shots)                 # kept for the final high-precision estimate
    theta = np.zeros(n_params)                                        # Hartree-Fock point
    history = []

    def evaluate(point, shots):
        energy, error = estimate_energy(hamiltonian, ansatz, point, sampler, shots)
        history.append(energy)
        return energy, error

    def affordable(shots, jobs_needed):
        remaining = sampler.remaining
        return (remaining["total_shots"] - jobs_needed * shots * n_groups >= final_reserve
                and remaining["jobs"] - jobs_needed >= 2
                and remaining["circuit_executions"] - jobs_needed * n_groups >= 2 * n_groups
                and time.monotonic() - start < 200)

    sweep_jobs = 1 + sum(2 * K for _, K in frequencies)               # centre point + fit points of every parameter
    n_sweeps, growth = N_SWEEPS, SHOT_GROWTH                          # shots per basis grow by `growth` every sweep
    weights = [growth ** s for s in range(n_sweeps)]
    optimisation_shots = total_shots - final_reserve
    for sweep in range(n_sweeps):
        shots = int(optimisation_shots * weights[sweep] / (sum(weights) * sweep_jobs * n_groups))
        if shots < 1 or not affordable(shots, sweep_jobs):
            break
        current, _ = evaluate(theta, shots)
        for k in rng.permutation(n_params):
            base, n_harmonics = frequencies[k]
            m, period = 2 * n_harmonics + 1, 2 * math.pi / base
            offsets = [period * j / m for j in range(m)]
            offsets = [o - period if o > period / 2 else o for o in offsets]
            values = [current]
            for offset in offsets[1:]:
                point = theta.copy()
                point[k] += offset
                values.append(evaluate(point, shots)[0])
            best_offset, best_value = _fit_minimum(offsets, values, base, n_harmonics)
            theta[k] += best_offset
            current = best_value                                     # fitted value reused as the next centre point

    # final estimate at the optimized parameters, using the shots that were kept in reserve
    remaining = sampler.remaining
    final_shots = max(1, min(remaining["total_shots"] // n_groups, 400_000))
    energy, error = evaluate(theta, final_shots)
    return {"ansatz": ansatz, "parameters": np.asarray(theta, dtype=float), "energy": float(energy),
            "std_error": float(error), "hamiltonian": hamiltonian, "history": [float(e) for e in history]}


def solve_geometry(integrals: dict, sampler: BudgetedSampler, seed: int) -> dict:
    """Complete workflow for one geometry; returns the dictionary described above."""
    hamiltonian = build_qubit_hamiltonian(integrals)      # 6-qubit Jordan-Wigner Hamiltonian (no tapering needed)
    return run_vqe(hamiltonian, sampler, seed)

#### Hints for Task VQE-T6

Open the hints one at a time. Each hint reveals more than the previous one.

<details>
<summary>Reveal Hint 1 (Conceptual direction)</summary>

Every energy you compute is noisy. Optimizers that rely on accurate gradients or line searches (for example BFGS) behave poorly with noisy evaluations; methods designed for noise, or methods that exploit the structure of the energy landscape, use the budget much better.

</details>

<details>
<summary>Reveal Hint 2 (Mathematical guidance)</summary>

If a parameter $t$ enters only rotation gates whose generators have eigenvalues spaced by integers, the energy along $t$ (with the other parameters fixed) is a trigonometric polynomial $a_0 + \sum_{k=1}^{K} (a_k \cos kt + b_k \sin kt)$. For a gate $e^{-itP}$ with a Pauli string $P$, $K = 2$ in $t$; $2K + 1$ evaluations determine the polynomial, and its minimum can be found exactly (Nakanishi et al., 2020; Ostaszewski et al., 2021). SPSA (Spall, 1992) is a general alternative with two evaluations per iteration.

</details>

<details>
<summary>Reveal Hint 3 (Implementation guidance)</summary>

Plan the budget before you code: (number of groups) $\times$ (shots per basis) $\times$ (number of jobs) must stay below $2 \times 10^{7}$. A schedule with fewer shots early and more shots late, plus one final high-precision estimate at the chosen parameters, works well. Starting from the Hartree-Fock parameters avoids wasting evaluations.

</details>

In [ ]:
# Provided Code: Do Not Modify
def _public_check_t6():
    sampler = BudgetedSampler(seed=SEED)
    start = time.monotonic()
    result = solve_geometry(EXAMPLE, sampler, SEED)
    elapsed = time.monotonic() - start
    required = {"ansatz", "parameters", "energy", "std_error", "hamiltonian", "history"}
    assert required <= set(result), f"missing keys: {required - set(result)}"
    error = 1000 * (result["energy"] - EXAMPLE["e_ref"])
    print(f"  estimate {result['energy']:.6f} +/- {result['std_error']:.6f} Ha "
          f"({error:+.2f} mHa from E_ref), usage {sampler.usage}, {elapsed:.0f} s")

ph.run_public_check("VQE-T6", _public_check_t6)

## 9. Experiments and analysis

Use the public geometries for the following experiments. Outside `solve_geometry` you may use exact tools (`Statevector`, `ph.noisy_density_matrix`) to analyze your results.

1. **Dissociation curve:** run `solve_geometry` for at least four public bond lengths, including 1.6 and 3.0 angstrom. Plot $E_{\mathrm{HF}}$, $E_{\mathrm{ref}}$ and your estimates $\hat{E} \pm \sigma_{\hat{E}}$ against $R$.
2. **Error decomposition:** for the same runs, compute the ideal and noisy energies of the final states (Section 5.4) and plot the three errors $E_{\mathrm{ideal}} - E_{\mathrm{ref}}$, $E_{\mathrm{noisy}} - E_{\mathrm{ref}}$ and $\hat{E} - E_{\mathrm{ref}}$ in millihartree on a logarithmic scale, with a line at chemical accuracy and the Hartree-Fock error for comparison.
3. **Convergence:** plot the optimization history at $R = 1.6$ angstrom.
4. **Resources:** report qubits, parameters, two-qubit gates, jobs, circuit executions, shots and wall-clock time per geometry.
5. **Ansatz comparison:** compare your ansatz with at least one alternative, in terms of the exact noiseless optimum and the noisy energy at that optimum.

In [ ]:
# Participant Code: Task VQE-X1
# Experiments for Section 9. Exact tools (Statevector, noisy density matrix) are used here, outside solve_geometry.
from scipy.optimize import minimize

KEYS = ("h1", "eri", "e_core", "n_orbitals", "n_electrons")          # hidden geometries come without e_hf / e_ref
runs = {}
for entry in PUBLIC_DATA:
    R = entry["bond_length_angstrom"]
    integrals = {k: entry[k] for k in KEYS}
    sampler = BudgetedSampler(seed=SEED)
    t0 = time.monotonic()
    result = solve_geometry(integrals, sampler, SEED)
    seconds = time.monotonic() - t0
    H = result["hamiltonian"]
    ansatz, theta = result["ansatz"], result["parameters"]
    e_ideal = Statevector(ansatz.assign_parameters(theta)).expectation_value(H).real
    e_noisy = float(np.real(np.trace(ph.noisy_density_matrix(ansatz, theta) @ H.to_matrix())))
    runs[R] = dict(entry=entry, result=result, usage=dict(sampler.usage), seconds=seconds,
                   ideal=e_ideal, noisy=e_noisy, cx=count_two_qubit_gates(ansatz))
mha = lambda x: 1000.0 * x
Rs = sorted(runs)

# 1. Dissociation curve ----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(Rs, [runs[R]["entry"]["e_hf"] for R in Rs], "s--", label="Hartree-Fock")
ax.plot(Rs, [runs[R]["entry"]["e_ref"] for R in Rs], "k-", label="exact (E_ref)")
ax.errorbar(Rs, [runs[R]["result"]["energy"] for R in Rs], yerr=[runs[R]["result"]["std_error"] for R in Rs],
            fmt="o", capsize=3, label="VQE estimate (noisy sampler)")
ax.set_xlabel("Li-H bond length (angstrom)"); ax.set_ylabel("Energy (hartree)"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()
print("Figure 1: LiH dissociation curve in the (2e, 3o) active space. The VQE estimates lie above the exact curve by "
      "the noise-induced shift (about 16-24 mHa) because the sampler runs the circuit under the official noise model.")

# 2. Error decomposition -------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.semilogy(Rs, [mha(runs[R]["entry"]["e_hf"] - runs[R]["entry"]["e_ref"]) for R in Rs], "s--", label="Hartree-Fock error")
ax.semilogy(Rs, [max(mha(runs[R]["ideal"] - runs[R]["entry"]["e_ref"]), 1e-3) for R in Rs], "o-", label="ideal state: E_ideal - E_ref")
ax.semilogy(Rs, [mha(runs[R]["noisy"] - runs[R]["entry"]["e_ref"]) for R in Rs], "^-", label="noisy state: E_noisy - E_ref")
ax.semilogy(Rs, [abs(mha(runs[R]["result"]["energy"] - runs[R]["entry"]["e_ref"])) for R in Rs], "v:", label="estimate: |E_hat - E_ref|")
ax.axhline(1.6, color="gray", ls="--", label="chemical accuracy (1.6 mHa)")
ax.set_xlabel("Li-H bond length (angstrom)"); ax.set_ylabel("error (mHa)"); ax.grid(alpha=0.3, which="both"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("Figure 2: errors in millihartree on a log scale. The ideal-state error stays near or below chemical accuracy; the noisy "
      "error is dominated by gate noise and is smaller than the Hartree-Fock error only for stretched bonds.")

# 3. Convergence at 1.6 angstrom -----------------------------------------------------------------------
r16 = runs[1.6]
history = np.array(r16["result"]["history"])
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(mha(history - r16["entry"]["e_ref"]), ".-", lw=0.6, label="energy estimate of each sampler job")
ax.axhline(mha(r16["noisy"] - r16["entry"]["e_ref"]), color="gray", ls="--", label="noisy energy at final parameters")
ax.set_xlabel("sampler job (energy evaluation)"); ax.set_ylabel("E_hat - E_ref (mHa)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Figure 3: energy estimates during the optimization at R = 1.6 angstrom. Each point is one sampler job; sweeps are "
      "visible as repeated scans of single parameters, and the energy settles at the noisy-state energy.")

# 4. Resources ---------------------------------------------------------------------------------------------
print("\nResources per geometry (limits: 6 qubits, 40 two-qubit gates, 600 jobs, 20000 executions, 2e7 shots, 300 s)")
print(" R(A)  qubits  params  cx   jobs  executions     shots      time(s)  E_ideal-E_ref  E_noisy-E_ref  E_hat-E_ref +/- sigma  (mHa)")
for R in Rs:
    r = runs[R]; res = r["result"]
    print(f" {R:.1f}   {res['hamiltonian'].num_qubits}      {len(res['parameters'])}      {r['cx']}   {r['usage']['jobs']:4d}  "
          f"{r['usage']['circuit_executions']:6d}   {r['usage']['total_shots']:10d}   {r['seconds']:5.1f}   "
          f"{mha(r['ideal'] - r['entry']['e_ref']):9.3f}   {mha(r['noisy'] - r['entry']['e_ref']):9.2f}   "
          f"{mha(res['energy'] - r['entry']['e_ref']):7.2f} +/- {mha(res['std_error']):.2f}")
full_hamiltonian = build_qubit_hamiltonian(EXAMPLE)
tapered = taper_hamiltonian(full_hamiltonian)
print(f"\nQubit Hamiltonian at 1.6 A: {len(full_hamiltonian)} Pauli terms on 6 qubits, {len(tapered)} terms on "
      f"{tapered.num_qubits} qubits after tapering; {len(group_qubitwise_commuting(full_hamiltonian))} qubit-wise "
      f"commuting measurement groups.")

# 5. Ansatz comparison at 1.6 angstrom (exact optimum and noisy energy at that optimum) -------------------
H16 = full_hamiltonian; H16_matrix = H16.to_matrix(); e_ref16 = EXAMPLE["e_ref"]


def hardware_efficient(layers):
    theta = ParameterVector("t", 6 * (layers + 1))
    circuit = QuantumCircuit(6)
    circuit.x(0); circuit.x(3)                                        # Hartree-Fock state
    for layer in range(layers):
        for q in range(6):
            circuit.ry(theta[6 * layer + q], q)
        for q in range(5):
            circuit.cx(q, q + 1)
    for q in range(6):
        circuit.ry(theta[6 * layers + q], q)
    return circuit


def reduced_natural_orbital():
    theta = ParameterVector("theta", 3)                                # pairing + one orbital rotation (1, 2)
    circuit = QuantumCircuit(6)
    circuit.x(0)
    circuit.append(XXPlusYYGate(theta[0], math.pi / 2), [0, 1]); circuit.append(XXPlusYYGate(theta[1], math.pi / 2), [1, 2])
    for k in range(3):
        circuit.cx(k, k + 3)
    circuit.append(XXPlusYYGate(theta[2], math.pi / 2), [1, 2]); circuit.append(XXPlusYYGate(theta[2], math.pi / 2), [4, 5])
    return circuit


candidates = {"natural-orbital (this work, 5 params)": build_ansatz(6),
              "natural-orbital reduced (3 params)": reduced_natural_orbital(),
              "hardware-efficient, 2 layers": hardware_efficient(2),
              "hardware-efficient, 3 layers": hardware_efficient(3)}
rng = np.random.default_rng(SEED)
print("\nAnsatz comparison at R = 1.6 angstrom (exact noiseless optimum, then the noisy energy at that optimum):")
print(f"{'ansatz':40s} params  cx   E_ideal-E_ref (mHa)   E_noisy-E_ref (mHa)")
for name, circuit in candidates.items():
    energy = lambda x: Statevector(circuit.assign_parameters(x)).expectation_value(H16).real
    n = circuit.num_parameters
    best = min((minimize(energy, np.zeros(n) if t == 0 else rng.uniform(-0.5, 0.5, n), method="BFGS") for t in range(4)),
               key=lambda r: r.fun)
    noisy = float(np.real(np.trace(ph.noisy_density_matrix(circuit, best.x) @ H16_matrix)))
    print(f"{name:40s} {n:4d}  {count_two_qubit_gates(circuit):3d}   {mha(best.fun - e_ref16):14.3f}   {mha(noisy - e_ref16):17.2f}")
print("Table: noiseless optimum and noisy energy at that optimum for each ansatz at R = 1.6 angstrom.")

**Your answer (VQE-Q1):**

LiH has 6 STO-3G spatial orbitals (12 spin orbitals), too many for a cheap exact treatment, so I freeze the Li 1s orbital (doubly occupied, inert) and drop the two π orbitals, which cannot mix with the σ orbitals by symmetry. This leaves the 3 σ orbitals with 2 electrons, i.e. 6 spin orbitals and 6 qubits in the Jordan–Wigner mapping with blocked ordering (qubit p = orbital p with spin α, qubit p+3 = spin β). The mapping gives 118 Pauli terms. The Hamiltonian conserves the numbers of α and β electrons, so the parities Z₀Z₁Z₂ and Z₃Z₄Z₅ commute with it. `Z2Symmetries.find_z2_symmetries` finds exactly these two generators; I evaluate each on the Hartree–Fock state (qubits 0 and 3 occupied), which gives eigenvalue −1 for both (odd α and odd β parity), and taper in that sector. The result has 4 qubits and 100 Pauli terms, and its lowest eigenvalue equals E_ref to better than 1e-6 Ha at all nine public geometries (checked). For the actual optimization I kept the 6-qubit Hamiltonian, because it is accepted by the grader and my ansatz is built for it; tapering would need the Hartree–Fock state and the ansatz to be mapped into the tapered frame. No trial state can go below E_ref: by the variational principle ⟨ψ|H|ψ⟩ ≥ E₀ for every normalized state, and the organizers verified that E_ref (the CASCI energy) is the lowest eigenvalue of the full 6-qubit Hamiltonian over all particle numbers (and of the tapered operator in the Hartree–Fock sector), so E_ref is the global minimum.

**Your answer (VQE-Q2):**

The exact ground state is a real singlet Σ_ij C_ij |i_α j_β⟩ with C a real symmetric 3×3 matrix, which can be written C = U diag(s) Uᵀ with an orthogonal U (natural orbitals). The ansatz builds exactly this: X on qubit 0 (Hartree–Fock), two Givens rotations (XXPlusYYGate with phase π/2 so amplitudes stay real) on the α qubits give s₀|100⟩ + s₁|010⟩ + s₂|001⟩ (2 parameters), three CNOTs (k → k+3) copy the electron to the β qubits and give Σ_k s_k |k_α k_β⟩, and three Givens rotations with the same angle on the α qubits and on the matching β qubits apply U (3 parameters). It has 5 parameters, which is exactly the dimension of the set of normalized real symmetric 3×3 matrices, so it conserves particle number and spin and can represent every real singlet in the active space, including the exact ground state. Its cost is 19 two-qubit gates after official transpilation (limit 40). Comparison at 1.6 Å: this ansatz reaches the exact energy (noiseless error 0.000 mHa) with a noisy-state error of 20.9 mHa; a reduced version with 3 parameters and 11 two-qubit gates reaches 0.36 mHa with 17.4 mHa noisy error; a hardware-efficient RY + CNOT-ladder ansatz with 2 layers (18 parameters, 10 two-qubit gates) only reaches 4.6 mHa, and with 3 layers (24 parameters, 15 two-qubit gates) 0.19 mHa with 16.0 mHa noisy error. The natural-orbital circuit therefore needs far fewer parameters (5 against 24) for the same noiseless accuracy, which matters for a shot-limited optimization; the price is a somewhat larger noise shift because of its 19 two-qubit gates.

**Your answer (VQE-Q3):**

The optimizer is sequential minimal optimization (Rotosolve generalised to the Givens gates). With the other parameters fixed, the energy as a function of one parameter is an exact trigonometric polynomial; scanning each parameter with Statevector (outside `solve_geometry`, for design only) showed that it has frequencies {1} for θ₀, {½, 1} for θ₁, {1, 2} for θ₂ and {½, 1, 3/2, 2} for θ₃ and θ₄, because of the half-angle dependence of the Givens gates and because α and β gates share an angle. For each parameter I measure the energy at 2K further equally spaced points (K = number of harmonics: 2, 4, 4, 8, 8), fit the polynomial exactly, and jump to its global minimum; the fitted minimum value is reused as the centre point of the next parameter. One sweep needs 27 jobs. I start at θ = 0 (Hartree–Fock) and run 6 sweeps whose shots per basis grow by a factor 1.6 per sweep (about 1 500 shots in the first sweep and 15 000 in the last), then spend the remaining 10 % of the shot budget on one final job at the returned parameters, so that the reported estimate comes from the sampler at exactly those parameters. The 27 measurement groups of the 118 terms are all measured in every job. Resources used per geometry: 163 jobs (limit 600), about 4 400–4 700 circuit executions (limit 20 000), 19 999 980–19 999 995 shots (limit 2·10⁷), about 3 s (limit 300 s), 19 two-qubit gates (limit 40); the table above lists each geometry. The final estimate has a standard error of 0.64–0.84 mHa (about 0.78 mHa at 1.6 Å), computed by adding the group variances, and the estimate agrees with the exact noisy energy of the final state within about two standard errors at every public geometry (for example 20.74 ± 0.78 mHa against 21.82 mHa at 1.6 Å), well inside the 4σ tolerance. The shot noise of the optimization itself is the main limit on the ideal-state accuracy: in tests, four times as many shots reduced the mean ideal error from 1.7 to 0.3 mHa, so with the real budget the result fluctuates between about 0.1 and 3 mHa from seed to seed.

**Your answer (VQE-Q4):**

With seed 2026 the ideal-state error is 0.11, 0.41, 1.46, 0.85, 0.62, 0.52, 0.33, 1.12 and 1.77 mHa at R = 1.0, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.6 and 3.0 Å, so chemical accuracy (1.6 mHa) of the ideal state is reached everywhere except the longest bond, where it is marginally missed (1.77 mHa); other seeds fluctuate by a few mHa because of shot noise, so I expect chemical accuracy in most but not all runs. Stretched bonds are harder because static correlation grows: the Hartree–Fock error rises from 14.9 mHa at 1.0 Å to 19.2 mHa at 1.6 Å, 36.9 mHa at 2.2 Å and 87.5 mHa at 3.0 Å, the ground state becomes a strong mixture of several configurations, so the starting point θ = 0 is worse, the energy landscape has more structure and, since the energy differences are larger, the optimizer has to move further. The noise shift is the dominant error of the noisy state: E_noisy − E_ref is 17.7–25.1 mHa for the 19 two-qubit gates, roughly 1 mHa per two-qubit gate (0.002 depolarizing probability times an energy scale of about 0.5 Ha), and it decreases somewhat at long bonds (to about 17.7 mHa at 3.0 Å). The noisy error is therefore never within 5 mHa, and it is below the Hartree–Fock error only for stretched bonds (R ≳ 2 Å). The 3-parameter variant cuts the gate count to 11 and the shift to about 17 mHa, but loses accuracy at long bonds. Zero-noise extrapolation could remove most of the shift: fold each CNOT (replace U by U U†U to scale the noise by 3, and by 5), measure the energy at noise factors 1, 3 and 5 and extrapolate linearly or quadratically to zero; the cost is 2–3 times the shots (and 19 → 57 or 95 two-qubit gates, which must stay below the gate limit in a real submission), plus amplified statistical error from the extrapolation, so with the present shot budget the accuracy gained would be partly lost to variance.

## 10. Expected deliverables

1. Working implementations of `build_qubit_hamiltonian`, `taper_hamiltonian`, `group_qubitwise_commuting`, `expectation_from_counts`, `estimate_energy`, `build_ansatz`, `run_vqe` and `solve_geometry`.
2. The experiments, tables and figures of Section 9.
3. Written answers to VQE-Q1, VQE-Q2, VQE-Q3 and VQE-Q4.

The notebook must execute from top to bottom in a fresh kernel within 30 minutes on a standard laptop, and each call of `solve_geometry` must finish within 300 seconds.

## 11. Published scoring rubric

| Criterion | Points | Assessment |
|:--|--:|:--|
| VQE-C1 Hamiltonian construction and Jordan-Wigner mapping (T1; public and undisclosed geometries) | 15 | automatic |
| VQE-C2 Symmetry reduction (T2) | 10 | automatic |
| VQE-C3 Measurement grouping (T3) | 8 | automatic |
| VQE-C4 Expectation values from counts (T4) | 7 | automatic |
| VQE-C5 Ansatz validity and two-qubit gate budget (T5, circuits returned at undisclosed geometries) | 10 | automatic |
| VQE-C6 Budget compliance and optimization integrity (T6) | 10 | automatic |
| VQE-C7 Ideal-state accuracy at undisclosed geometries | 15 | automatic, formula below |
| VQE-C8 Noisy-hardware accuracy at undisclosed geometries | 10 | automatic, formula below |
| VQE-C9 Reported estimate and uncertainty | 5 | automatic |
| VQE-C10 Scientific interpretation, experiments and figures (Section 9, Q1 to Q4) | 10 | manual, structured review form |
| **Total** | **100** | |

**How the hidden evaluation works.** For each of the two undisclosed geometries $g$, the grader calls `solve_geometry(integrals_g, sampler, seed)` twice, each time with a fresh `BudgetedSampler` and an independent random seed, so that four runs are evaluated in total and a single unlucky run has limited weight. From each returned dictionary it computes, outside your notebook:

* the validity of the returned Hamiltonian: either the 6-qubit Jordan-Wigner Hamiltonian of the given integrals (any spin-orbital ordering with identical spectrum is accepted), or a reduced operator whose eigenvalues are all eigenvalues of that Hamiltonian and whose lowest eigenvalue equals $E_{\mathrm{ref}}$ within $10^{-6}$ hartree;
* $\Delta E_{\mathrm{ideal},g} = E_{\mathrm{ideal}} - E_{\mathrm{ref}}$ and $\Delta E_{\mathrm{noisy},g} = E_{\mathrm{noisy}} - E_{\mathrm{ref}}$, from exact statevector and exact noisy density-matrix simulations of the returned circuit and parameters with the returned Hamiltonian;
* the Hartree-Fock error $\Delta E_{\mathrm{HF},g} = E_{\mathrm{HF}} - E_{\mathrm{ref}}$ of that geometry.

**Accuracy scores.** With $\ell(x) = \log_{10}(x)$, errors in millihartree and $\operatorname{clip}$ restricting a value to $[0, 1]$:
$$
s_{\mathrm{ideal},g} = \operatorname{clip}\!\left(\frac{\ell(\Delta E_{\mathrm{HF},g}) - \ell(\max(\Delta E_{\mathrm{ideal},g},\, 1.6))}{\ell(\Delta E_{\mathrm{HF},g}) - \ell(1.6)}\right),
\qquad
s_{\mathrm{noisy},g} = \operatorname{clip}\!\left(\frac{\ell(\Delta E_{\mathrm{HF},g}) - \ell(\max(\Delta E_{\mathrm{noisy},g},\, 5.0))}{\ell(\Delta E_{\mathrm{HF},g}) - \ell(5.0)}\right).
$$
VQE-C7 $= 15 \times \bar{s}_{\mathrm{ideal}}$ and VQE-C8 $= 10 \times \bar{s}_{\mathrm{noisy}}$, where the bar denotes the average over the four runs. Full credit therefore requires chemical accuracy of the ideal state and a noisy energy within 5 mHa; a result no better than Hartree-Fock earns nothing.

**Other automatic criteria (per run, averaged over the four runs).** VQE-C5: circuit without measurements whose qubit count matches the returned Hamiltonian and whose parameter count matches `"parameters"` (40 percent), and at most 40 two-qubit gates after official transpilation (60 percent). VQE-C6: the call completes without an exception or `BudgetExceededError` within the time limit (40 percent), the final parameters were evaluated by the sampler (40 percent), and the sampler's counters are within the budget (20 percent). VQE-C9: `"std_error"` is positive and finite and $\lvert \hat{E} - E_{\mathrm{noisy}} \rvert \le \max(4\sigma_{\hat{E}},\, 1\ \mathrm{mHa})$.

**Penalties and edge cases (fixed before the event).** An invalid Hamiltonian, a circuit that exceeds the gate budget, an exception, a `BudgetExceededError` or a timeout gives zero points for VQE-C7, VQE-C8 and VQE-C9 in that run. An ideal energy more than $10^{-6}$ hartree below $E_{\mathrm{ref}}$ indicates an inconsistent submission; it receives zero points for VQE-C7 and VQE-C8 in that run and is flagged for judge review. Computing energies inside `solve_geometry` by any route other than the sampler is a rule violation that is checked during code review of all top-ranked submissions.

**Tie-breakers** for the overall ranking, applied in order: (1) higher Challenge 3 score; (2) lower mean $\Delta E_{\mathrm{ideal}}$ at the undisclosed geometries; (3) fewer two-qubit gates in the returned circuits (mean over the runs); (4) fewer total shots used at the undisclosed geometries; (5) higher Challenge 2 score. Teams that are still tied share the rank.

The general rules on re-execution, partial credit and integrity violations stated in Challenge 1, Section 11, apply here as well.

## 12. Submission instructions

1. Restart the kernel and run all cells; check that the run completes within the time limit.
2. Keep the file name `03_vqe_molecular_energy.ipynb`.
3. Submit it in your team folder as described in `README.md`.

## 13. Scientific references

1. A. Peruzzo, J. McClean, P. Shadbolt, M.-H. Yung, X.-Q. Zhou, P. J. Love, A. Aspuru-Guzik and J. L. O'Brien, "A variational eigenvalue solver on a photonic quantum processor," *Nature Communications* 5, 4213 (2014). DOI: 10.1038/ncomms5213
2. J. R. McClean, J. Romero, R. Babbush and A. Aspuru-Guzik, "The theory of variational hybrid quantum-classical algorithms," *New Journal of Physics* 18, 023023 (2016). DOI: 10.1088/1367-2630/18/2/023023
3. A. Kandala, A. Mezzacapo, K. Temme, M. Takita, M. Brink, J. M. Chow and J. M. Gambetta, "Hardware-efficient variational quantum eigensolver for small molecules and quantum magnets," *Nature* 549, 242 (2017). DOI: 10.1038/nature23879
4. P. Jordan and E. Wigner, "Uber das Paulische Aquivalenzverbot," *Zeitschrift fur Physik* 47, 631 (1928). DOI: 10.1007/BF01331938
5. S. Bravyi, J. M. Gambetta, A. Mezzacapo and K. Temme, "Tapering off qubits to simulate fermionic Hamiltonians," arXiv: 1701.08213 (2017).
6. V. Verteletskyi, T.-C. Yen and A. F. Izmaylov, "Measurement optimization in the variational quantum eigensolver using a minimum clique cover," *Journal of Chemical Physics* 152, 124114 (2020). DOI: 10.1063/1.5141458
7. K. M. Nakanishi, K. Fujii and S. Todo, "Sequential minimal optimization for quantum-classical hybrid algorithms," *Physical Review Research* 2, 043158 (2020). DOI: 10.1103/PhysRevResearch.2.043158
8. M. Ostaszewski, E. Grant and M. Benedetti, "Structure optimization for parameterized quantum circuits," *Quantum* 5, 391 (2021). DOI: 10.22331/q-2021-01-28-391
9. J. C. Spall, "Multivariate stochastic approximation using a simultaneous perturbation gradient approximation," *IEEE Transactions on Automatic Control* 37, 332 (1992). DOI: 10.1109/9.119632
10. K. Temme, S. Bravyi and J. M. Gambetta, "Error mitigation for short-depth quantum circuits," *Physical Review Letters* 119, 180509 (2017). DOI: 10.1103/PhysRevLett.119.180509
11. A. Kandala, K. Temme, A. D. Corcoles, A. Mezzacapo, J. M. Chow and J. M. Gambetta, "Error mitigation extends the computational reach of a noisy quantum processor," *Nature* 567, 491 (2019). DOI: 10.1038/s41586-019-1040-7
12. Q. Sun et al., "PySCF: the Python-based simulations of chemistry framework," *WIREs Computational Molecular Science* 8, e1340 (2018). DOI: 10.1002/wcms.1340
13. A. Szabo and N. S. Ostlund, *Modern Quantum Chemistry: Introduction to Advanced Electronic Structure Theory*, Dover (1996).
14. T. Helgaker, P. Jorgensen and J. Olsen, *Molecular Electronic-Structure Theory*, Wiley (2000).
15. Qiskit documentation: https://quantum.cloud.ibm.com/docs/en/api/qiskit
16. Qiskit Aer documentation: https://qiskit.github.io/qiskit-aer/